In [1]:
# ============================================
# 03_market_api.ipynb
# Fetches live exchange rates and exports them
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import requests
from datetime import date

# --- Setup shared project directory ---
PROJECT_ROOT = "/content/drive/MyDrive/project"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/output", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/quotations", exist_ok=True)

# --- Step 1: Config ---
FRANKFURTER_BASE_URL = "https://api.frankfurter.dev/v1/latest"

# Adjust based on currencies actually used in your supplier sheet / quotations
currencies_needed = ["USD", "EUR", "GBP"]

# Fallback rates in case the live API fails during a demo (update with real current values)
fallback_rates = {
    "USD": 96.46,
    "EUR": 110.05,
    "GBP": 129.92
}

# --- Step 2: Fetch function with retries ---
def get_exchange_rate(base="USD", target="INR", retries=3, backoff=1.5):
    params = {"base": base, "symbols": target}

    for attempt in range(retries):
        try:
            response = requests.get(FRANKFURTER_BASE_URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            return {
                "base_currency": base,
                "target_currency": target,
                "exchange_rate": data["rates"][target],
                "date": data["date"]
            }
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed for {base}->{target}: {e}")
            if attempt < retries - 1:
                time.sleep(backoff ** attempt)
            else:
                raise

# --- Step 3: Safe wrapper with fallback ---
def get_exchange_rate_safe(base="USD", target="INR", fallback_rate=None):
    try:
        return get_exchange_rate(base, target)
    except Exception as e:
        print(f"⚠️ API failed for {base}->{target}, using fallback: {e}")
        if fallback_rate is not None:
            return {
                "base_currency": base,
                "target_currency": target,
                "exchange_rate": fallback_rate,
                "date": str(date.today()),
                "source": "fallback"
            }
        raise

# --- Step 4: Fetch all needed rates ---
rates = [
    get_exchange_rate_safe(base=cur, target="INR", fallback_rate=fallback_rates.get(cur))
    for cur in currencies_needed
]

# --- Step 5: Validate ---
def validate_rate(rate_entry):
    assert rate_entry["exchange_rate"] is not None, "Rate missing"
    assert isinstance(rate_entry["exchange_rate"], (int, float)), "Rate not numeric"
    assert rate_entry["exchange_rate"] > 0, "Rate not positive"
    return True

print("\n=== Validation ===")
for r in rates:
    try:
        validate_rate(r)
        print(f"✅ {r['base_currency']}→{r['target_currency']}: {r['exchange_rate']} valid")
    except AssertionError as e:
        print(f"❌ {r['base_currency']}→{r['target_currency']}: {e}")

# --- Step 6: Preview results ---
print("\n=== Fetched Rates ===")
for r in rates:
    print(json.dumps(r, indent=2))

# --- Step 7: Export as JSON for downstream notebooks ---
with open(f"{PROJECT_ROOT}/output/exchange_rates.json", "w") as f:
    json.dump(rates, f, indent=2)

print(f"\n✅ Saved {len(rates)} exchange rate entries to {PROJECT_ROOT}/output/exchange_rates.json")

Mounted at /content/drive

=== Validation ===
✅ USD→INR: 96.45 valid
✅ EUR→INR: 110.1985 valid
✅ GBP→INR: 129.82 valid

=== Fetched Rates ===
{
  "base_currency": "USD",
  "target_currency": "INR",
  "exchange_rate": 96.45,
  "date": "2026-07-20"
}
{
  "base_currency": "EUR",
  "target_currency": "INR",
  "exchange_rate": 110.1985,
  "date": "2026-07-20"
}
{
  "base_currency": "GBP",
  "target_currency": "INR",
  "exchange_rate": 129.82,
  "date": "2026-07-20"
}

✅ Saved 3 exchange rate entries to /content/drive/MyDrive/project/output/exchange_rates.json
